# 01a — Assemble the AA-identity modeling dataset (TEM-1 / Firnberg)

Builds the model-ready table the EDA (`01`) and benchmark (`02`) both read from disk. This is the
**assembly** step: parse the raw Firnberg 2014 deep mutational scan into one tidy row per single missense
variant, validate every boundary, and write the processed checkpoint.

Runs in run order **after** `01_EDA` in numbering but is a prerequisite for it — kept as `01a` so the
lightweight dataset build sits with the EDA it feeds. Idempotent: skips the write if the checkpoint already
exists and matches.

**Input:** `data/raw/BLAT_ECOLX_Firnberg_2014.csv` + `.fasta` (immutable).
**Output:** `data/processed/traditional_ml_aa_identity/modeling_dataset.{parquet,csv}` + `README.json`.

In [1]:
# self-contained: resolve project root via .projectroot, read from disk
import sys, re, json
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root))
from paths import *   # RAW, PROCESSED, ...
import pandas as pd, numpy as np
print("root:", root)

root: /Users/kdave2/Beta-Lactamase Mutagenesis/1 - ML


In [2]:
# 1. load raw firnberg table + wild-type sequence
# raw cols: mutant (e.g. H24S), mutated_sequence, DMS_score, DMS_score_bin
raw = pd.read_csv(RAW/"BLAT_ECOLX_Firnberg_2014.csv")
wt_seq = None
for i, line in enumerate((RAW/"BLAT_ECOLX_Firnberg_2014.fasta").read_text().splitlines()):
    if line.strip() == ">WT":
        wt_seq = None  # next line holds the sequence
        wt_lines = []
        break
# parse fasta: >WT record then its sequence line(s)
lines = (RAW/"BLAT_ECOLX_Firnberg_2014.fasta").read_text().splitlines()
for i, l in enumerate(lines):
    if l.strip() == ">WT":
        wt_seq = lines[i+1].strip(); break
assert wt_seq and len(wt_seq) > 0, "WT sequence not found in FASTA"
print("raw rows:", len(raw), "| WT length:", len(wt_seq))

raw rows: 4783 | WT length: 286


In [3]:
# 2. parse each mutant id (wt_aa, position, mut_aa) and assemble the modeling columns
m = raw["mutant"].str.extract(r"^([A-Z])(\d+)([A-Z])$")
m.columns = ["wt_aa", "position", "mut_aa"]
assert m["wt_aa"].notna().all(), f"unparseable mutant ids: {raw['mutant'][m['wt_aa'].isna()].tolist()[:5]}"
m["position"] = m["position"].astype(int)

df = pd.DataFrame({
    "seq_id": raw["mutant"],
    "wt_seq": wt_seq,
    "mut_seq": raw["mutated_sequence"],
    "wt_aa": m["wt_aa"],
    "mut_aa": m["mut_aa"],
    "position": m["position"],
    "DMS_score": raw["DMS_score"],
    "DMS_score_bin": raw["DMS_score_bin"],
})
print("assembled:", df.shape, "| columns:", list(df.columns))
df.head(3)

assembled: (4783, 8) | columns: ['seq_id', 'wt_seq', 'mut_seq', 'wt_aa', 'mut_aa', 'position', 'DMS_score', 'DMS_score_bin']


In [4]:
# 3. validate at the boundary (CLAUDE.md: assert, don't warn) — stop on any failure
# 3a. wt_aa matches the WT sequence at its position (1-indexed)
bad = df[df.apply(lambda r: r.wt_seq[r.position-1] != r.wt_aa, axis=1)]
assert len(bad) == 0, f"{len(bad)} rows where wt_aa != wt_seq[pos-1]: {bad.seq_id.tolist()[:5]}"

# 3b. mut_seq is exactly one substitution from wt_seq, at `position`, changing wt_aa->mut_aa
def single_sub(r):
    diffs = [i for i,(a,b) in enumerate(zip(r.wt_seq, r.mut_seq)) if a != b]
    return (len(diffs)==1 and diffs[0]==r.position-1
            and r.wt_seq[diffs[0]]==r.wt_aa and r.mut_seq[diffs[0]]==r.mut_aa)
ok = df.apply(single_sub, axis=1)
assert ok.all(), f"{(~ok).sum()} rows fail the single-substitution check: {df.seq_id[~ok].tolist()[:5]}"

# 3c. shape / label / uniqueness invariants
assert (df.wt_seq.str.len() == df.mut_seq.str.len()).all()
assert set(df.DMS_score_bin.unique()) == {0, 1}
assert df.notna().all().all()
assert df.seq_id.is_unique
assert len(df) == len(raw)
print("all boundary validations passed")
print("class balance:", df.DMS_score_bin.value_counts().to_dict(),
      "| positions:", df.position.nunique())

all boundary validations passed
class balance: {1: 2397, 0: 2386} | positions: 263


In [5]:
# 4. deterministic ordering + write the processed checkpoint (idempotent)
df = df.sort_values(["position","mut_aa"], kind="mergesort").reset_index(drop=True)
OUTDIR = PROCESSED/"traditional_ml_aa_identity"; OUTDIR.mkdir(parents=True, exist_ok=True)
pq = OUTDIR/"modeling_dataset.parquet"

if pq.exists():
    prev = pd.read_parquet(pq)
    if prev.equals(df):
        print("checkpoint already up to date — skipping write:", pq.relative_to(BASE_DIR))
    else:
        df.to_parquet(pq, index=False); df.to_csv(OUTDIR/"modeling_dataset.csv", index=False)
        print("checkpoint changed — rewrote:", pq.relative_to(BASE_DIR))
else:
    df.to_parquet(pq, index=False); df.to_csv(OUTDIR/"modeling_dataset.csv", index=False)
    print("wrote new checkpoint:", pq.relative_to(BASE_DIR))

checkpoint already up to date — skipping write: data/processed/traditional_ml_aa_identity/modeling_dataset.parquet


In [6]:
# 5. write dataset metadata alongside the checkpoint
meta = dict(
    source="data/raw/BLAT_ECOLX_Firnberg_2014.csv + .fasta (>WT)",
    n_rows=int(len(df)), n_positions=int(df.position.nunique()),
    position_range=[int(df.position.min()), int(df.position.max())],
    wt_seq_len=len(wt_seq),
    label="DMS_score_bin (1=functional, 0=non-functional); positive class = functional",
    class_balance={str(k): int(v) for k,v in df.DMS_score_bin.value_counts().items()},
    columns=list(df.columns),
    validation="wt_aa==wt_seq[pos-1]; exactly one substitution at pos (wt_aa->mut_aa); binary label; no NaN; unique seq_id; row count matches raw",
    ordering="sorted by position then mut_aa (deterministic)",
)
(OUTDIR/"README.json").write_text(json.dumps(meta, indent=2))
print("wrote metadata:", (OUTDIR/"README.json").relative_to(BASE_DIR))
print(json.dumps(meta, indent=2))

wrote metadata: data/processed/traditional_ml_aa_identity/README.json
{
  "source": "data/raw/BLAT_ECOLX_Firnberg_2014.csv + .fasta (>WT)",
  "n_rows": 4783,
  "n_positions": 263,
  "position_range": [
    24,
    286
  ],
  "wt_seq_len": 286,
  "label": "DMS_score_bin (1=functional, 0=non-functional); positive class = functional",
  "class_balance": {
    "1": 2397,
    "0": 2386
  },
  "columns": [
    "seq_id",
    "wt_seq",
    "mut_seq",
    "wt_aa",
    "mut_aa",
    "position",
    "DMS_score",
    "DMS_score_bin"
  ],
  "validation": "wt_aa==wt_seq[pos-1]; exactly one substitution at pos (wt_aa->mut_aa); binary label; no NaN; unique seq_id; row count matches raw",
  "ordering": "sorted by position then mut_aa (deterministic)"
}
